Kita coba lihat apa isi dataset nya

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('dataset/train.csv')
df

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,0,42,technician,married,secondary,no,7,no,no,cellular,25,aug,117,3,-1,0,unknown,0
1,1,38,blue-collar,married,secondary,no,514,no,no,unknown,18,jun,185,1,-1,0,unknown,0
2,2,36,blue-collar,married,secondary,no,602,yes,no,unknown,14,may,111,2,-1,0,unknown,0
3,3,27,student,single,secondary,no,34,yes,no,unknown,28,may,10,2,-1,0,unknown,0
4,4,26,technician,married,secondary,no,889,yes,no,cellular,3,feb,902,1,-1,0,unknown,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
749995,749995,29,services,single,secondary,no,1282,no,yes,unknown,4,jul,1006,2,-1,0,unknown,1
749996,749996,69,retired,divorced,tertiary,no,631,no,no,cellular,19,aug,87,1,-1,0,unknown,0
749997,749997,50,blue-collar,married,secondary,no,217,yes,no,cellular,17,apr,113,1,-1,0,unknown,0
749998,749998,32,technician,married,secondary,no,-274,no,no,cellular,26,aug,108,6,-1,0,unknown,0


Dataset ini adalah kumpulan dari data pelanggan yang dihubungi oleh BANK dalam kampanye tahun ini tentang deposito berjangka mereka.
- kolom job mungkin nanti bisa kita cari tahu pekerjaan apa aja yang paling sering subscribe term deposito (tapi ini additional work aja)
- kolom default untuk melihat apakah seseorang punya kartu kredit atau enggak
- kolom balance merupakan jumlah rata rata saldo tahunan dalam kurs euro
- kolom housing & loan merupakan boolean apakah seseorang punya pinjaman perumahan atau pinjaman pribadi
- contact adalah media yang digunakan bank untuk menghubungi pelanggan pada kampanye ini
- day adalah tanggal contact, month adalah bulan terjadinya kapan, dan duration adalah durasi call nya
- kolom campaign adalah total contact yang dilakukan pada pelanggan pada periode kampanye tahun ini, sedangkan kolom previous adalah total contact yang dilakuikan pada periode kampanye tahun tahun sebelumnya
- pdays adalah jjumlah hari sejak pelanggan dihubungi dari kampanye tahun sebelumnya, sampai dihubungi lagi pada kampenye tahun ini, kalau belum pernah dihubungi pada kampanye sebelumnya, maka nilainya -1
- kolom outcome adalah hasil dari kampanye tahun sebelumnya terhadap pelanggan tersebut, kalau pada tahun sebelumnya tidak pernah dihubungi oleh pihak bank (pdays = -1) maka sepertinya sejalan dengan poutcome unknown (hipotesis awal). Kategorinya adalah unknown/other/failure/success'
- kolom y menyatakan apakah pelanggan tersebut jadi melakukan subscription atau tidak

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 18 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   id         750000 non-null  int64 
 1   age        750000 non-null  int64 
 2   job        750000 non-null  object
 3   marital    750000 non-null  object
 4   education  750000 non-null  object
 5   default    750000 non-null  object
 6   balance    750000 non-null  int64 
 7   housing    750000 non-null  object
 8   loan       750000 non-null  object
 9   contact    750000 non-null  object
 10  day        750000 non-null  int64 
 11  month      750000 non-null  object
 12  duration   750000 non-null  int64 
 13  campaign   750000 non-null  int64 
 14  pdays      750000 non-null  int64 
 15  previous   750000 non-null  int64 
 16  poutcome   750000 non-null  object
 17  y          750000 non-null  int64 
dtypes: int64(9), object(9)
memory usage: 103.0+ MB


dari info yang didapatkan tidak terdapat missing value, yang berarti datanya berisi semua, sehingga tidak perlu kita lakukan penanganan missing value, yang perlu kita lakukan cek duplikat aja nanti.

eksplorasi data

sebelum kubagi datanya ke kolom kategorikal dan numerikal, aku mau lihat nilai unique dari setiap kolom dulu

In [4]:
# mendapatkan list semua kolom
columns = df.columns

# cek nilai unik dari setiap kolom
for col in columns:
    unique_values = df[col].unique()
    num_unique = len(unique_values)
    if num_unique >= 20:
        print(f"kolom '{col}' punya >= 20 nilai unik\n")
    else:
        print(f"kolom '{col}' memiliki nilai unik:\n{df[col].unique()}\n")

kolom 'id' punya >= 20 nilai unik

kolom 'age' punya >= 20 nilai unik

kolom 'job' memiliki nilai unik:
['technician' 'blue-collar' 'student' 'admin.' 'management' 'entrepreneur'
 'self-employed' 'unknown' 'services' 'retired' 'housemaid' 'unemployed']

kolom 'marital' memiliki nilai unik:
['married' 'single' 'divorced']

kolom 'education' memiliki nilai unik:
['secondary' 'primary' 'tertiary' 'unknown']

kolom 'default' memiliki nilai unik:
['no' 'yes']

kolom 'balance' punya >= 20 nilai unik

kolom 'housing' memiliki nilai unik:
['no' 'yes']

kolom 'loan' memiliki nilai unik:
['no' 'yes']

kolom 'contact' memiliki nilai unik:
['cellular' 'unknown' 'telephone']

kolom 'day' punya >= 20 nilai unik

kolom 'month' memiliki nilai unik:
['aug' 'jun' 'may' 'feb' 'apr' 'nov' 'jul' 'jan' 'oct' 'mar' 'sep' 'dec']

kolom 'duration' punya >= 20 nilai unik

kolom 'campaign' punya >= 20 nilai unik

kolom 'pdays' punya >= 20 nilai unik

kolom 'previous' punya >= 20 nilai unik

kolom 'poutcome' memi

In [5]:
df_1 = df[df['pdays']==-1].groupby('poutcome').size().reset_index(name='count')
df_1

,poutcome,count
0,failure,12
1,other,3
2,success,3
3,unknown,672416


dari hipotesis awal kita, dapat diketahui bahwa memang untuk pelanggan yang belum pernah dihubungi pada kampanye sebelumnya, hasil outcome nya adalah unknown. Data ini bisa kubuat nanti menjadi kolom baru, pada saat feature engineering. Rencananya akan dibuat kolom baru 'was-contacted_before' dengan nilai true/false.

In [6]:
a = df['pdays'].value_counts()
a

pdays
-1      672434
 182      2515
 92       2275
 183      2074
 181      1698
         ...  
 759         1
 529         1
 794         1
 617         1
 376         1
Name: count, Length: 596, dtype: int64

In [9]:
# membuat tabel baru, jika pdays != -1 maka nilainya akan 1, dan jika pdays == -1 maka nilainya akan 0
df['was_contacted_before'] = (df['pdays'] != -1).astype(int)
df['was_contacted_before'].value_counts()

was_contacted_before
0    672434
1     77566
Name: count, dtype: int64

In [11]:
# untuk kolom pdays, aku akan coba ubah -1 jadi 0, angka yang lebih dapat dibaca oleh model ketimbang kalau kita pakai -1
df['pdays_modified'] = df['pdays'].replace(-1, 0)
df['pdays_modified'].value_counts().sort_index()

pdays_modified
0      672435
1          45
2         210
3           1
4           8
        ...  
838         4
842         5
850         1
854         2
871         2
Name: count, Length: 595, dtype: int64

In [12]:
df_0 = df[df['previous']==0].groupby('pdays').size().reset_index(name='count')
df_0

,pdays,count
0,-1,672417
1,0,1
2,87,1
3,100,1
4,101,1
5,119,1
6,188,1
7,194,1
8,336,1
9,342,1


penemuan di atas dapat menjadi kunci penyederhanaan data kita. kita punya hubungan seperti ini:
- previous = 0
- pdays = -1
- poutcome = 'unknown'

informasinya tumpang tindih/redundant, jadi bisa kita buat fitur baru nanti.